In [1]:
# First, reconnect to your graph (run the setup cells from main notebook first)
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
import pandas as pd

load_dotenv()


c:\dev\Diabetes_AgenticGraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
graph = Neo4jGraph(
    url="neo4j+s://bbb47553.databases.neo4j.io",
    username="neo4j",
    password="jU3ptukapvL5orYyfqiCr2I8rCuPVpBGDA8JRsvSye8",
    database="neo4j"
)

def run_query(query, description=""):
    if description:
        print(f"\n{'='*80}")
        print(f"Query: {description}")
        print(f"{'='*80}")
    try:
        result = graph.query(query)
        if result:
            df = pd.DataFrame(result)
            print(df.to_string(index=False))
            print(f"\n✓ Found {len(result)} results")
        else:
            print("⚠ No results found")
        return result
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

print("✓ Setup complete!")

✓ Setup complete!


In [3]:
type(graph)

langchain_neo4j.graphs.neo4j_graph.Neo4jGraph

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key = os.getenv("GROQ_API_KEY"),model_name="llama-3.3-70b-versatile")

In [5]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

chain= GraphCypherQAChain.from_llm(graph = graph, llm = llm, verbose =True, allow_dangerous_requests = True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x00000241719C0E90>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False

In [11]:
print(graph.query("Find 20 Therapeutic Agents (Medications)"))

CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'Find': expected 'ALTER', 'ORDER BY', 'CALL', 'USING PERIODIC COMMIT', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE' or 'WITH' (line 1, column 1 (offset: 0))
"Find 20 Therapeutic Agents (Medications)"
 ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

## Query 1: Find Diagnostic Tests (FIXED)

## Query 1: Find Diagnostic Tests (FIXED)

In [2]:
# FIXED: Use 'id' instead of 'name'
query = """
MATCH (n:DiagnosticTest)
RETURN n.id as diagnostic_test
LIMIT 20
"""
run_query(query, "Find all Diagnostic Tests")


Query: Find all Diagnostic Tests
                                           diagnostic_test
                                  blood glucose monitoring
                   screening for complications of diabetes
                                                  A1C test
                                   systolic blood pressure
                                  diastolic blood pressure
                                 screening for retinopathy
                                        waist-to-hip ratio
                          bioelectrical impedance analysis
                  Screening for micronutrient deficiencies
                   Insulin pump therapy without automation
                        Automated insulin delivery systems
                                   hybrid closed-loop pump
Insulin pump with threshold/predictive low-glucose suspend
           continuous subcutaneous insulin infusion (CSII)
                       Continuous glucose monitoring (CGM)
                      

[{'diagnostic_test': 'blood glucose monitoring'},
 {'diagnostic_test': 'screening for complications of diabetes'},
 {'diagnostic_test': 'A1C test'},
 {'diagnostic_test': 'systolic blood pressure'},
 {'diagnostic_test': 'diastolic blood pressure'},
 {'diagnostic_test': 'screening for retinopathy'},
 {'diagnostic_test': 'waist-to-hip ratio'},
 {'diagnostic_test': 'bioelectrical impedance analysis'},
 {'diagnostic_test': 'Screening for micronutrient deficiencies'},
 {'diagnostic_test': 'Insulin pump therapy without automation'},
 {'diagnostic_test': 'Automated insulin delivery systems'},
 {'diagnostic_test': 'hybrid closed-loop pump'},
 {'diagnostic_test': 'Insulin pump with threshold/predictive low-glucose suspend'},
 {'diagnostic_test': 'continuous subcutaneous insulin infusion (CSII)'},
 {'diagnostic_test': 'Continuous glucose monitoring (CGM)'},
 {'diagnostic_test': 'sensor-augmented pump'},
 {'diagnostic_test': 'closed-loop therapy'},
 {'diagnostic_test': 'SMBG'},
 {'diagnostic_test'

## Query 2: Find A1C and Glucose Related Entities

In [3]:
query = """
MATCH (n)
WHERE n.id IS NOT NULL
  AND (toLower(n.id) CONTAINS 'a1c' 
    OR toLower(n.id) CONTAINS 'glucose' 
    OR toLower(n.id) CONTAINS 'hemoglobin')
RETURN DISTINCT labels(n) as node_types, n.id as entity_id
LIMIT 30
"""
run_query(query, "Find A1C and Glucose-related entities")


Query: Find A1C and Glucose-related entities
                                 node_types                                                  entity_id
                   [__Entity__, TargetGoal]                                              glucose goals
[__Entity__, DiagnosticTest, ScreeningTest]                                   blood glucose monitoring
                                    [Chunk]                   ba1c2de68704c6dc1353423ad746eb1f77b67bc7
                  [__Entity__, MetricValue]                                               A1C was 8.1%
                  [__Entity__, MetricValue]                               A1C (mean difference −0.21%)
               [__Entity__, DiagnosticTest]                                                   A1C test
                  [__Entity__, MetricValue]                                               A1C was 9.1%
                   [__Entity__, TargetGoal]                                          A1C goal of <7.5%
                  [__Entity

[{'node_types': ['__Entity__', 'TargetGoal'], 'entity_id': 'glucose goals'},
 {'node_types': ['__Entity__', 'DiagnosticTest', 'ScreeningTest'],
  'entity_id': 'blood glucose monitoring'},
 {'node_types': ['Chunk'],
  'entity_id': 'ba1c2de68704c6dc1353423ad746eb1f77b67bc7'},
 {'node_types': ['__Entity__', 'MetricValue'], 'entity_id': 'A1C was 8.1%'},
 {'node_types': ['__Entity__', 'MetricValue'],
  'entity_id': 'A1C (mean difference −0.21%)'},
 {'node_types': ['__Entity__', 'DiagnosticTest'], 'entity_id': 'A1C test'},
 {'node_types': ['__Entity__', 'MetricValue'], 'entity_id': 'A1C was 9.1%'},
 {'node_types': ['__Entity__', 'TargetGoal'],
  'entity_id': 'A1C goal of <7.5%'},
 {'node_types': ['__Entity__', 'MetricValue'],
  'entity_id': '−0.47% change in A1C'},
 {'node_types': ['__Entity__', 'MetricValue'],
  'entity_id': '−0.5% change in A1C'},
 {'node_types': ['__Entity__', 'MetricValue'], 'entity_id': 'A1C (−0.5%)'},
 {'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'In

## Query 3: Find Medications

In [4]:
query = """
MATCH (n:TherapeuticAgent)
RETURN n.id as medication
LIMIT 30
"""
run_query(query, "Find all Therapeutic Agents (Medications)")


Query: Find all Therapeutic Agents (Medications)
                                                medication
                                antihypertensive treatment
treatment with a goal diastolic blood pressure of <90 mmHg
        intensive systolic blood pressure goal (<120 mmHg)
                                perindopril and indapamide
                                                       ARB
                                             ACE inhibitor
                                                    Statin
                         PCSK9 monoclonal antibody therapy
                                                 ezetimibe
                                           PCSK9 inhibitor
                                    lipid-lowering therapy
                                            statin therapy
                         moderate-intensity statin therapy
                             high-intensity statin therapy
             PCSK9 inhibitor therapy with inclisiran siRNA
      

[{'medication': 'antihypertensive treatment'},
 {'medication': 'treatment with a goal diastolic blood pressure of <90 mmHg'},
 {'medication': 'intensive systolic blood pressure goal (<120 mmHg)'},
 {'medication': 'perindopril and indapamide'},
 {'medication': 'ARB'},
 {'medication': 'ACE inhibitor'},
 {'medication': 'Statin'},
 {'medication': 'PCSK9 monoclonal antibody therapy'},
 {'medication': 'ezetimibe'},
 {'medication': 'PCSK9 inhibitor'},
 {'medication': 'lipid-lowering therapy'},
 {'medication': 'statin therapy'},
 {'medication': 'moderate-intensity statin therapy'},
 {'medication': 'high-intensity statin therapy'},
 {'medication': 'PCSK9 inhibitor therapy with inclisiran siRNA'},
 {'medication': 'bempedoic acid therapy'},
 {'medication': 'GLP-1 RAs'},
 {'medication': 'thiazide-like diuretics'},
 {'medication': 'dihydropyridine calcium channel blockers'},
 {'medication': 'SGLT2 inhibitor'},
 {'medication': 'Finerenone'},
 {'medication': 'calcium channel blocker'},
 {'medication'

In [5]:
# Search for specific medications
query = """
MATCH (n)
WHERE n.id IS NOT NULL
  AND (toLower(n.id) CONTAINS 'metformin' 
    OR toLower(n.id) CONTAINS 'insulin'
    OR toLower(n.id) CONTAINS 'glp'
    OR toLower(n.id) CONTAINS 'sglt')
RETURN DISTINCT labels(n) as node_types, n.id as entity_id
LIMIT 30
"""
run_query(query, "Search for specific medications")


Query: Search for specific medications
                    node_types                                                                                                                                       entity_id
     [__Entity__, MetricValue]                                                                                                                                    insulin dose
  [__Entity__, PatientProfile] U.S. adults prescribed GLP-1 RAs between 2017 and 2022, in which more than half had obesity or overweight and more than 80% had type 2 diabetes
  [__Entity__, DiagnosticTest]                                                                                                         Insulin pump therapy without automation
  [__Entity__, DiagnosticTest]                                                                                                              Automated insulin delivery systems
  [__Entity__, DiagnosticTest]                                                       

[{'node_types': ['__Entity__', 'MetricValue'], 'entity_id': 'insulin dose'},
 {'node_types': ['__Entity__', 'PatientProfile'],
  'entity_id': 'U.S. adults prescribed GLP-1 RAs between 2017 and 2022, in which more than half had obesity or overweight and more than 80% had type 2 diabetes'},
 {'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'Insulin pump therapy without automation'},
 {'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'Automated insulin delivery systems'},
 {'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'Insulin pump with threshold/predictive low-glucose suspend'},
 {'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'continuous subcutaneous insulin infusion (CSII)'},
 {'node_types': ['__Entity__', 'MetricValue'], 'entity_id': 'insulin doses'},
 {'node_types': ['__Entity__', 'PatientProfile'],
  'entity_id': 'people with diabetes treated with insulin therapy'},
 {'node_types': ['__Entity__', 'PatientProfile'],
  'en

## Query 4: Find Complications

In [6]:
query = """
MATCH (n:Complication)
RETURN n.id as complication
LIMIT 30
"""
run_query(query, "Find all Complications")


Query: Find all Complications
                          complication
                          dyslipidemia
                   severe hypoglycemia
                                   CKD
atherosclerotic cardiovascular disease
                                   CVD
                           CKD Stage 3
                       nonfatal stroke
                           nonfatal MI
                 cardiovascular events
                           eye disease
           microvascular complications
                        kidney disease
                  autonomic neuropathy
                                stroke
                         heart failure
                                    MI
                  cardiovascular death
               history of pancreatitis
                     prior ASCVD event
          Chronic kidney disease (CKD)
             Acute kidney injury (AKI)
               eGFR <60 mL/min/1.73 m2
                         CVD mortality
                          drop in

[{'complication': 'dyslipidemia'},
 {'complication': 'severe hypoglycemia'},
 {'complication': 'CKD'},
 {'complication': 'atherosclerotic cardiovascular disease'},
 {'complication': 'CVD'},
 {'complication': 'CKD Stage 3'},
 {'complication': 'nonfatal stroke'},
 {'complication': 'nonfatal MI'},
 {'complication': 'cardiovascular events'},
 {'complication': 'eye disease'},
 {'complication': 'microvascular complications'},
 {'complication': 'kidney disease'},
 {'complication': 'autonomic neuropathy'},
 {'complication': 'stroke'},
 {'complication': 'heart failure'},
 {'complication': 'MI'},
 {'complication': 'cardiovascular death'},
 {'complication': 'history of pancreatitis'},
 {'complication': 'prior ASCVD event'},
 {'complication': 'Chronic kidney disease (CKD)'},
 {'complication': 'Acute kidney injury (AKI)'},
 {'complication': 'eGFR <60 mL/min/1.73 m2'},
 {'complication': 'CVD mortality'},
 {'complication': 'drop in eGFR'},
 {'complication': 'increased albuminuria'},
 {'complication':

In [7]:
# Search for specific complications
query = """
MATCH (n)
WHERE n.id IS NOT NULL
  AND (toLower(n.id) CONTAINS 'kidney'
    OR toLower(n.id) CONTAINS 'retinopathy'
    OR toLower(n.id) CONTAINS 'neuropathy'
    OR toLower(n.id) CONTAINS 'cardiovascular'
    OR toLower(n.id) CONTAINS 'ckd')
RETURN DISTINCT labels(n) as node_types, n.id as entity_id
LIMIT 30
"""
run_query(query, "Search for complication-related entities")


Query: Search for complication-related entities
                            node_types                                               entity_id
          [__Entity__, DiagnosticTest]                               screening for retinopathy
               [__Entity__, Condition]                                  cardiovascular disease
               [__Entity__, Condition]                                  kidney transplantation
             [__Entity__, MetricValue]                      cardiovascular and kidney outcomes
 [__Entity__, Condition, Complication]                                                     CKD
              [__Entity__, TargetGoal]                           cardiovascular risk reduction
              [__Entity__, TargetGoal]                              slowing progression of CKD
 [__Entity__, Condition, Complication]                  atherosclerotic cardiovascular disease
            [__Entity__, Complication]                                             CKD Stage 3
 

[{'node_types': ['__Entity__', 'DiagnosticTest'],
  'entity_id': 'screening for retinopathy'},
 {'node_types': ['__Entity__', 'Condition'],
  'entity_id': 'cardiovascular disease'},
 {'node_types': ['__Entity__', 'Condition'],
  'entity_id': 'kidney transplantation'},
 {'node_types': ['__Entity__', 'MetricValue'],
  'entity_id': 'cardiovascular and kidney outcomes'},
 {'node_types': ['__Entity__', 'Condition', 'Complication'],
  'entity_id': 'CKD'},
 {'node_types': ['__Entity__', 'TargetGoal'],
  'entity_id': 'cardiovascular risk reduction'},
 {'node_types': ['__Entity__', 'TargetGoal'],
  'entity_id': 'slowing progression of CKD'},
 {'node_types': ['__Entity__', 'Condition', 'Complication'],
  'entity_id': 'atherosclerotic cardiovascular disease'},
 {'node_types': ['__Entity__', 'Complication'], 'entity_id': 'CKD Stage 3'},
 {'node_types': ['__Entity__', 'RiskFactor'],
  'entity_id': 'elevated cardiovascular risk'},
 {'node_types': ['__Entity__', 'RiskFactor'],
  'entity_id': 'cardiov

## Query 5: Explore Actual Relationships

In [8]:
# Get relationships between actual entities (not chunks)
query = """
MATCH (a)-[r]->(b)
WHERE a.id IS NOT NULL AND b.id IS NOT NULL
  AND NOT 'Chunk' IN labels(a)
  AND NOT 'Chunk' IN labels(b)
RETURN labels(a)[0] as from_type, 
       a.id as from_entity,
       type(r) as relationship,
       labels(b)[0] as to_type,
       b.id as to_entity
LIMIT 30
"""
run_query(query, "Sample entity relationships (excluding chunks)")


Query: Sample entity relationships (excluding chunks)
 from_type              from_entity relationship    to_type                                                                                                         to_entity
__Entity__             insulin dose   APPLIES_TO __Entity__                                                                                       adults with type 1 diabetes
__Entity__             data metrics   APPLIES_TO __Entity__                                                                                    adults aged 18 years and older
__Entity__             data metrics   APPLIES_TO __Entity__                                                                     adults with more than a high school education
__Entity__             data metrics   APPLIES_TO __Entity__                                                                           children, adolescents, and young adults
__Entity__             data metrics   APPLIES_TO __Entity__                

[{'from_type': '__Entity__',
  'from_entity': 'insulin dose',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  'to_entity': 'adults with type 1 diabetes'},
 {'from_type': '__Entity__',
  'from_entity': 'data metrics',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  'to_entity': 'adults aged 18 years and older'},
 {'from_type': '__Entity__',
  'from_entity': 'data metrics',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  'to_entity': 'adults with more than a high school education'},
 {'from_type': '__Entity__',
  'from_entity': 'data metrics',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  'to_entity': 'children, adolescents, and young adults'},
 {'from_type': '__Entity__',
  'from_entity': 'data metrics',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  'to_entity': 'non-Hispanic Black individuals'},
 {'from_type': '__Entity__',
  'from_entity': 'data metrics',
  'relationship': 'APPLIES_TO',
  'to_type': '__Entity__',
  '

## Query 6: Find Specific Clinical Patterns

In [9]:
# Find what treats complications
query = """
MATCH (agent:TherapeuticAgent)-[:TREATS]->(comp:Complication)
RETURN agent.id as treatment, comp.id as complication
LIMIT 20
"""
run_query(query, "What treats what complications?")


Query: What treats what complications?
                                                 treatment                complication
                                antihypertensive treatment       cardiovascular events
                                antihypertensive treatment microvascular complications
treatment with a goal diastolic blood pressure of <90 mmHg                      stroke
treatment with a goal diastolic blood pressure of <90 mmHg                          MI
treatment with a goal diastolic blood pressure of <90 mmHg        cardiovascular death
        intensive systolic blood pressure goal (<120 mmHg)             nonfatal stroke
        intensive systolic blood pressure goal (<120 mmHg)                 nonfatal MI
        intensive systolic blood pressure goal (<120 mmHg)                      stroke
        intensive systolic blood pressure goal (<120 mmHg)               heart failure
        intensive systolic blood pressure goal (<120 mmHg)                          MI
   

[{'treatment': 'antihypertensive treatment',
  'complication': 'cardiovascular events'},
 {'treatment': 'antihypertensive treatment',
  'complication': 'microvascular complications'},
 {'treatment': 'treatment with a goal diastolic blood pressure of <90 mmHg',
  'complication': 'stroke'},
 {'treatment': 'treatment with a goal diastolic blood pressure of <90 mmHg',
  'complication': 'MI'},
 {'treatment': 'treatment with a goal diastolic blood pressure of <90 mmHg',
  'complication': 'cardiovascular death'},
 {'treatment': 'intensive systolic blood pressure goal (<120 mmHg)',
  'complication': 'nonfatal stroke'},
 {'treatment': 'intensive systolic blood pressure goal (<120 mmHg)',
  'complication': 'nonfatal MI'},
 {'treatment': 'intensive systolic blood pressure goal (<120 mmHg)',
  'complication': 'stroke'},
 {'treatment': 'intensive systolic blood pressure goal (<120 mmHg)',
  'complication': 'heart failure'},
 {'treatment': 'intensive systolic blood pressure goal (<120 mmHg)',
  'com

In [10]:
# Find what increases risk of complications
query = """
MATCH (rf:RiskFactor)-[:INCREASES_RISK]->(comp:Complication)
RETURN rf.id as risk_factor, comp.id as complication
LIMIT 20
"""
run_query(query, "What risk factors increase complications?")


Query: What risk factors increase complications?
                 risk_factor                           complication
            excess adiposity                 diabetes complications
                  overweight                        type 2 diabetes
                  overweight                         GDM recurrence
                     Obesity                             neuropathy
                dyslipidemia atherosclerotic cardiovascular disease
                dyslipidemia                retinopathy progression
  postprandial hyperglycemia                             macrosomia
                Hypertension                                    CKD
                Hypertension                                    CVD
                Hypertension                            CKD Stage 3
                Hypertension                  cardiovascular events
                Hypertension                        CKD progression
                Hypertension                   Diabetic Retinopath

[{'risk_factor': 'excess adiposity', 'complication': 'diabetes complications'},
 {'risk_factor': 'overweight', 'complication': 'type 2 diabetes'},
 {'risk_factor': 'overweight', 'complication': 'GDM recurrence'},
 {'risk_factor': 'Obesity', 'complication': 'neuropathy'},
 {'risk_factor': 'dyslipidemia',
  'complication': 'atherosclerotic cardiovascular disease'},
 {'risk_factor': 'dyslipidemia', 'complication': 'retinopathy progression'},
 {'risk_factor': 'postprandial hyperglycemia', 'complication': 'macrosomia'},
 {'risk_factor': 'Hypertension', 'complication': 'CKD'},
 {'risk_factor': 'Hypertension', 'complication': 'CVD'},
 {'risk_factor': 'Hypertension', 'complication': 'CKD Stage 3'},
 {'risk_factor': 'Hypertension', 'complication': 'cardiovascular events'},
 {'risk_factor': 'Hypertension', 'complication': 'CKD progression'},
 {'risk_factor': 'Hypertension', 'complication': 'Diabetic Retinopathy'},
 {'risk_factor': 'Dyslipidemia', 'complication': 'CVD'},
 {'risk_factor': 'Dyslipi

In [11]:
# Find diagnostic tests and what they detect
query = """
MATCH (test:DiagnosticTest)-[:DIAGNOSES]->(condition)
RETURN test.id as diagnostic_test, 
       labels(condition) as condition_type,
       condition.id as condition
LIMIT 20
"""
run_query(query, "What do diagnostic tests diagnose?")


Query: What do diagnostic tests diagnose?
                                diagnostic_test                                                    condition_type                  condition
        screening for complications of diabetes                                           [__Entity__, Condition]  complications of diabetes
                                       A1C test                 [__Entity__, Condition, Complication, RiskFactor]            type 2 diabetes
                                       A1C test                 [__Entity__, Condition, Complication, RiskFactor]            type 1 diabetes
                      screening for retinopathy                             [__Entity__, Condition, Complication]                retinopathy
                             waist-to-hip ratio                               [__Entity__, Condition, RiskFactor]           excess adiposity
               bioelectrical impedance analysis                               [__Entity__, Condition, RiskFacto

[{'diagnostic_test': 'screening for complications of diabetes',
  'condition_type': ['__Entity__', 'Condition'],
  'condition': 'complications of diabetes'},
 {'diagnostic_test': 'A1C test',
  'condition_type': ['__Entity__', 'Condition', 'Complication', 'RiskFactor'],
  'condition': 'type 2 diabetes'},
 {'diagnostic_test': 'A1C test',
  'condition_type': ['__Entity__', 'Condition', 'Complication', 'RiskFactor'],
  'condition': 'type 1 diabetes'},
 {'diagnostic_test': 'screening for retinopathy',
  'condition_type': ['__Entity__', 'Condition', 'Complication'],
  'condition': 'retinopathy'},
 {'diagnostic_test': 'waist-to-hip ratio',
  'condition_type': ['__Entity__', 'Condition', 'RiskFactor'],
  'condition': 'excess adiposity'},
 {'diagnostic_test': 'bioelectrical impedance analysis',
  'condition_type': ['__Entity__', 'Condition', 'RiskFactor'],
  'condition': 'excess adiposity'},
 {'diagnostic_test': 'Screening for micronutrient deficiencies',
  'condition_type': ['__Entity__', 'Con

In [13]:
# Find target goals and what they apply to
query = """
MATCH (goal:TargetGoal)-[:APPLIES_TO]->(profile:PatientProfile)
RETURN goal.id as target_goal, profile.id as patient_profile
LIMIT 20
"""
run_query(query, "What target goals apply to which patient profiles?")


Query: What target goals apply to which patient profiles?
target_goal                            patient_profile
No A1C goal Older adults with very complex/poor health
No A1C goal    Any adults with limited life expectancy
weight loss                adults with type 1 diabetes
weight loss                       people with diabetes

✓ Found 4 results


[{'target_goal': 'No A1C goal',
  'patient_profile': 'Older adults with very complex/poor health'},
 {'target_goal': 'No A1C goal',
  'patient_profile': 'Any adults with limited life expectancy'},
 {'target_goal': 'weight loss',
  'patient_profile': 'adults with type 1 diabetes'},
 {'target_goal': 'weight loss', 'patient_profile': 'people with diabetes'}]

## Query 7: Browse All Entity Types

In [14]:
# Sample entities from each type
for entity_type in ['DiagnosticTest', 'Condition', 'MetricValue', 'TargetGoal', 
                     'PatientProfile', 'Complication', 'RiskFactor', 'ScreeningTest',
                     'TherapeuticAgent', 'ReferralCriteria', 'ScreeningFrequency']:
    query = f"""
    MATCH (n:{entity_type})
    RETURN n.id as entity
    LIMIT 5
    """
    print(f"\n{'='*80}")
    print(f"{entity_type}:")
    print(f"{'='*80}")
    result = graph.query(query)
    if result:
        for r in result:
            print(f"  - {r['entity']}")
    else:
        print("  (none found)")


DiagnosticTest:
  - blood glucose monitoring
  - screening for complications of diabetes
  - A1C test
  - systolic blood pressure
  - diastolic blood pressure

Condition:
  - alcohol-related harms
  - complications of diabetes
  - excess adiposity
  - overweight
  - Obesity

MetricValue:
  - insulin dose
  - data metrics
  - time in range
  - 135–499 mg/dL
  - A1C was 8.1%

TargetGoal:
  - glucose goals
  - live free from the burden of discrimination
  - general treatment goals
  - 20–30%
  - 14 g of fiber/1,000 kcal

PatientProfile:
  - adults aged 18 years and older
  - adults with more than a high school education
  - children, adolescents, and young adults
  - non-Hispanic Black individuals
  - individuals aged 19 years and under

Complication:
  - dyslipidemia
  - severe hypoglycemia
  - CKD
  - atherosclerotic cardiovascular disease
  - CVD

RiskFactor:
  - excess adiposity
  - overweight
  - Obesity
  - Overweight
  - nutritional deficiencies

ScreeningTest:
  - blood glucose m

Testing natual language to cyper queries 

In [41]:
# THEN import from the module
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_openai import ChatOpenAI


from langchain_core.prompts import PromptTemplate

# ============================================================================
# CYPHER GENERATION PROMPT
# ============================================================================

CYPHER_GENERATION_TEMPLATE = """Task:
Generate a Cypher query for the Type 2 Diabetes Clinical Knowledge Graph stored in Neo4j.

Context:
This knowledge graph contains clinical guidelines from the ADA Standards of Care, organized into 5 main categories:
1. Assessment & Diagnosis (diagnostic tests, screening, glycemic targets)
2. Patient Education & Lifestyle (DSMES, nutrition, SDOH, behavioral interventions)
3. Pharmacology & Technology (medications, CGM, insulin pumps, treatment algorithms)
4. Complications Management (CVD, CKD, retinopathy, neuropathy, foot care)
5. Special Populations (children, older adults, pregnancy)

Schema:
CATEGORY_SCHEMAS

Instructions:
1. Use ONLY the relationship types and properties provided in the schema above
2. Do NOT use any other relationship types or properties not explicitly listed
3. Ensure relationship directions match the schema
4. Always alias entities and relationships for clarity (e.g., `d:DiagnosticTest`, `r:TREATS`)
5. Use WITH clauses to chain query segments when needed
6. For division operations, filter denominators to be non-zero to prevent errors
7. Do NOT generate queries that INSERT, CREATE, DELETE, or MERGE data - READ-ONLY queries only
8. Use the `id` property for entity identification (NOT `name`)
9. When querying patient populations, use appropriate filters

Clinical Query Guidelines:
- For diagnostic queries: Link DiagnosticTest → Condition
- For treatment queries: Link TherapeuticAgent → Complication
- For complications: Link RiskFactor → Complication → TherapeuticAgent
- For screening: Link ScreeningTest → Condition
- For patient profiles: Link TargetGoal → PatientProfile

Important Notes:
- Entity identification: ALL entities use the `id` property (e.g., `n.id = "A1C test"`)
- String matching: Use `toLower()` for case-insensitive searches
- Filtering: Use WHERE clauses with `id` property
- Multi-hop queries: Leverage relationship chains for complex clinical logic

Response Format:
- Return ONLY the Cypher query
- Do NOT include explanations, apologies, or markdown formatting
- Do NOT include comments in the Cypher code
- Ensure the query is syntactically correct and executable

The question is:
{question}
"""

cypher_generation_prompt = PromptTemplate(
    input_variables=[CATEGORY_SCHEMAS, "question"],
    template=CYPHER_GENERATION_TEMPLATE
)


# ============================================================================
# QA GENERATION PROMPT (for answering based on Cypher results)
# ============================================================================

QA_GENERATION_TEMPLATE = """You are an assistant that helps answer questions about Type 2 Diabetes clinical guidelines based on data from a Neo4j knowledge graph.

Use the following context from the database query to answer the question. If the context doesn't contain enough information to answer the question, say "I don't have enough information to answer that question."

Context from database:
{context}

Question: {question}

Provide a clear, concise answer based on the context. If there are specific clinical values, medications, or recommendations, include them in your answer.

Answer:"""

qa_generation_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=QA_GENERATION_TEMPLATE
)


# ============================================================================
# CATEGORY-SPECIFIC QUERY HINTS (for reference)
# ============================================================================

CATEGORY_QUERY_HINTS = {
    "assessment_diagnosis": """
    Key patterns:
    - (DiagnosticTest)-[:DIAGNOSES]->(Condition)
    - (DiagnosticTest)-[:SCREENS_FOR]->(Condition)
    - (TargetGoal)-[:APPLIES_TO]->(PatientProfile)
    """,
    
    "patient_education_lifestyle": """
    Key patterns:
    - (Intervention)-[:RECOMMENDS]->(Behavior)
    - (Behavior)-[:IMPROVES]->(Outcome)
    - (ScreeningTest)-[:ASSESSES]->(Condition)
    """,
    
    "pharmacology_technology": """
    Key patterns:
    - (TherapeuticAgent)-[:TREATS]->(Complication)
    - (TherapeuticAgent)-[:HAS_RISK]->(AdverseEvent)
    - (DiagnosticTest)-[:MEASURES]->(MetricValue)
    """,
    
    "complications_management": """
    Key patterns:
    - (RiskFactor)-[:INCREASES_RISK]->(Complication)
    - (TherapeuticAgent)-[:TREATS]->(Complication)
    - (ScreeningTest)-[:DETECTS]->(Complication)
    - (Complication)-[:TRIGGERS_REFERRAL]->(ReferralCriteria)
    """,
    
    "special_populations": """
    Key patterns:
    - (TargetGoal)-[:APPLIES_TO]->(PatientProfile)
    - (TherapeuticAgent)-[:CONTRAINDICATED_IN]->(PatientProfile)
    - (PatientProfile)-[:REQUIRES]->(SpecialConsideration)
    """
}

CATEGORY_SCHEMAS= {
    "assessment_diagnosis": {
        "nodes": [
            "DiagnosticTest",
            "Condition", 
            "MetricValue",
            "ScreeningFrequency",
            "TargetGoal",
            "PatientProfile"
        ],
        "relationships": [
            "DIAGNOSES",
            "REQUIRES_SCREENING",
            "HAS_TARGET",
            "APPLIES_TO",
            "DEFINED_BY",
            "MEASURES"
        ]
    },
    "patient_education_lifestyle": {
        "nodes": [
            "Intervention",
            "Behavior",
            "SocialDeterminant",
            "Outcome",
            "ScreeningTool"
        ],
        "relationships": [
            "RECOMMENDS",
            "IMPROVES",
            "BARRIER_TO",
            "ASSESSES",
            "MODIFIES",
            "INFLUENCES",
            "LEADS_TO",
            "PREVENTS"
        ]
    },
    "pharmacology_technology": {
        "nodes": [
            "MedicationClass",
            "ActiveIngredient",
            "Device",
            "ClinicalIndication",
            "AdverseEvent",
            "Administration",
            "Dosage"
        ],
        "relationships": [
            "CONTAINS",
            "TREATS",
            "CAUSES",
            "MONITORS",
            "REQUIRES_EDUCATION",
            "CONTRAINDICATED_WITH",
            "TITRATED_BY",
            "INDICATED_FOR"
        ]
    },
    "complications_management": {
        "nodes": [
            "Complication",
            "ScreeningTest",
            "TherapeuticAgent",
            "RiskFactor",
            "ReferralCriteria"
        ],
        "relationships": [
            "SCREENS_FOR",
            "TREATS",
            "INCREASES_RISK",
            "REQUIRES_REFERRAL",
            "DETECTED_BY",
            "MANAGED_BY",
            "REQUIRES_MONITORING"
        ]
    },
    "special_populations": {
        "nodes": [
            "PopulationSegment",
            "SpecificGoal",
            "AllowedMedication",
            "ContraindicatedMedication",
            "AgeRange"
        ],
        "relationships": [
            "HAS_GOAL",
            "ALLOWS",
            "CONTRAINDICATES",
            "APPLIES_TO",
            "HAS_TARGET_OVERRIDE",
            "CAN_USE",
            "MUST_AVOID",
            "REQUIRES_DEINTENSIFICATION"
        ]
    }
}


In [42]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

In [43]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    graph=graph,
    verbose=True,
    validate_cypher=True,
    qa_prompt=qa_generation_prompt,
    cypher_prompt=cypher_generation_prompt,
    top_k=10,
    allow_dangerous_requests=True,  # Required for newer LangChain versions
)

In [44]:
test_questions = [
    "What medications are used to treat cardiovascular complications?",
    "What are the A1C target goals for different patient populations?",
    "Which risk factors increase the likelihood of diabetic retinopathy?",
]
response = cypher_chain.invoke({"query": test_questions[0]})
print("Answer:", response['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Complication {id: "CVD"})<-[:TREATS]-(t:TherapeuticAgent) RETURN t.id
Full Context:
[{'t.id': 'ARB'}, {'t.id': 'ACE inhibitor'}, {'t.id': 'Statin'}]

> Finished chain.
Answer: The medications used to treat cardiovascular complications include ARBs (Angiotensin Receptor Blockers), ACE inhibitors (Angiotensin-Converting Enzyme inhibitors), and Statins.
